# FarmCam retraining (natural colours)

Retrains the FarmCam models on the colour-fixed datasets from `prepare_dataset.py`:
- **pest** model: YOLOv8n detection, classes `whitefly`, `worm`
- **fungus** model: YOLOv8n-seg segmentation, class `fungus`

**Before you start**
1. *Runtime → Change runtime type → T4 GPU.*
2. In Google Drive, make a folder **FarmCam** and upload:
   - `datasets_fixed.zip` (from `C:\Users\eru\Documents\Thesis`)
   - `best_Pest.pt` and `best_segmentation.pt` (your current models; training starts from them)
3. Choose which models to train in the **Settings** cell (`TRAIN = ...`).
4. *Runtime → Run all.* Results are saved to `FarmCam/runs` in Drive, so a disconnect loses nothing.

Keep the same model sizes (YOLOv8n): the Pi's Hailo scripts and post-processing expect them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/FarmCam'
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls -la "$DRIVE"

In [ ]:
# Same Ultralytics version as offline testing on the PC, so results compare 1:1.
!pip install -q ultralytics==8.4.183 onnx onnxsim
!rm -rf /content/data && mkdir -p /content/data
!unzip -q "$DRIVE/datasets_fixed.zip" -d /content/data
!ls /content/data /content/data/pest /content/data/fungus
!cat /content/data/pest/data.yaml /content/data/fungus/data.yaml

## Settings

- `TRAIN`: which models to train. Leave out `'fungus'` until its Roboflow labels are finished and exported; training it on the old labels gains little.
- `fliplr` / `flipud`: insects and lesions look the same mirrored or upside down.
- `hsv_h` stays small: colour is a real clue for worms and fungus. Never add channel-swap augmentation; that is the bug we just removed.
- `close_mosaic`: the last epochs train on whole photos, like the camera sees them.
- `patience`: stops early when validation stops improving.

In [ ]:
from ultralytics import YOLO

# Real machine photos (prepare_dataset.py --machine-pest/--machine-fungus):
# pest_v2 = whitefly/worm, fungus_v4 = fungus + 'dirt' (fly droppings, soil).
# pest_v3 = pest_v2 as a gentle fine-tune: pest_v2's automatic SGD lr 0.01 knocked
# the trained model back, so "best" stayed at epoch 2 and early stopping ended it.
# pest_v4 = pest_v3 + the 65 whiteflies confirmed in review_whiteflies.html (they
# were on photos marked Null, so v3 learned them as background).
# The whitefly classifier at the end of the notebook trains in the same run.
TRAIN = ['pest_v4']

MODELS = {
    'pest':   dict(start='best_Pest.pt',         data='/content/data/pest/data.yaml',   out='best_detect'),
    'fungus': dict(start='best_segmentation.pt', data='/content/data/fungus/data.yaml', out='best_seg'),
    # from your old fungus model / from fresh YOLOv8n-seg weights (downloaded automatically)
    'fungus_v2':   dict(start='best_segmentation.pt', data='/content/data/fungus/data.yaml', out='best_seg_v2'),
    'fungus_coco': dict(start='yolov8n-seg.pt',       data='/content/data/fungus/data.yaml', out='best_seg_coco'),
    'fungus_v3':   dict(start='best_seg_v2_natural.pt', data='/content/data/fungus/data.yaml', out='best_seg_v3'),
    'pest_v2':     dict(start='best_detect_natural.pt', data='/content/data/pest/data.yaml', out='best_detect_v2'),
    'fungus_v4':   dict(start='best_seg_v2_natural.pt', data='/content/data/fungus/data.yaml', out='best_seg_v4'),
    'pest_v3':     dict(start='best_detect_natural.pt', data='/content/data/pest/data.yaml', out='best_detect_v3'),
    'pest_v4':     dict(start='best_detect_v3_natural.pt', data='/content/data/pest/data.yaml', out='best_detect_v4'),
}
# Per-model extras on top of COMMON. No copy_paste for fungus: it pastes lesions
# onto the background and taught round 1 to see fungus off the leaf.
GENTLE = dict(epochs=80, patience=0, optimizer='AdamW', lr0=0.0005, lrf=0.1, warmup_epochs=1, close_mosaic=10)
EXTRA = {
    'fungus':      dict(epochs=150, patience=30),
    'fungus_v2':   dict(epochs=150, patience=30),
    'fungus_coco': dict(epochs=200, patience=40),
    'fungus_v3':   dict(epochs=150, patience=30),
    'pest_v2':     dict(epochs=150, patience=30),
    'fungus_v4':   dict(epochs=150, patience=30),
    # Small fixed learning rate, every epoch runs (patience=0); last.pt is
    # exported too, because "best" is judged mostly on worms.
    'pest_v3':     GENTLE,
    'pest_v4':     GENTLE,
}
COMMON = dict(imgsz=640, epochs=120, patience=25, batch=16, workers=2,
              hsv_h=0.01, hsv_s=0.5, hsv_v=0.4, fliplr=0.5, flipud=0.5,
              degrees=10, scale=0.4, mosaic=1.0, close_mosaic=15,
              project=f'{DRIVE}/runs', exist_ok=True, plots=True)
print('Will train:', TRAIN)

In [ ]:
import os
# If Colab disconnected mid-training, set RESUME = True and run this cell again:
# it continues from the last saved epoch in Drive instead of starting over.
RESUME = False

for name in TRAIN:
    m = MODELS[name]
    last = f'{DRIVE}/runs/{name}/weights/last.pt'
    if RESUME:
        try:
            YOLO(last).train(resume=True)
        except Exception as e:      # already finished (or never started)
            print(name, 'not resumed:', e)
    else:
        start = f"{DRIVE}/{m['start']}"
        start = start if os.path.exists(start) else m['start']   # e.g. yolov8n-seg.pt
        YOLO(start).train(data=m['data'], name=name,
                                            **{**COMMON, **EXTRA.get(name, {})})

## Check on the test split

These are model-only numbers (mAP). The real check is `evaluate.py` on the PC, which also runs the Pi's crops and verifier checks.

In [ ]:
for name in TRAIN:
    model = YOLO(f'{DRIVE}/runs/{name}/weights/best.pt')
    metrics = model.val(data=MODELS[name]['data'], split='test', imgsz=640, plots=False)
    print(name, 'test mAP50 =', round(metrics.box.map50, 3), ' mAP50-95 =', round(metrics.box.map, 3))
    if hasattr(metrics, 'seg'):
        print(name, 'mask mAP50 =', round(metrics.seg.map50, 3))

In [ ]:
# ONNX for the Hailo compiler (same settings as best na pt/export_onnx.py).
# Both best.pt and last.pt: the PC's evaluate.py decides which one goes on the Pi.
import shutil
for name in TRAIN:
    out = MODELS[name]['out']
    for weights, suffix in (('best', ''), ('last', '_last')):
        pt = f'{DRIVE}/runs/{name}/weights/{weights}.pt'
        path = YOLO(pt).export(format='onnx', imgsz=640, opset=11, simplify=True, dynamic=False, half=False)
        shutil.copy(path, f'{DRIVE}/{out}{suffix}.onnx')
        shutil.copy(pt, f'{DRIVE}/{out}{suffix}_natural.pt')
!ls -la "$DRIVE"

## Next, on the PC

Download the new `*_natural.pt` files from Drive into `C:\Users\eru\Documents\Thesis\models_new\` and compare them with the current models using `evaluate.py` (same photos, same Pi pipeline). Only if they are better: compile the ONNX files to HEF and deploy.

## Whitefly classifier (2nd deep-learning stage)

Trains the small network that re-checks every whitefly candidate on a sharp, zoomed crop (`wf_classifier.py` on the Pi). It replaces the old colour rules. It runs after the detection models in **Settings** (`TRAIN`), in the same *Run all*; about 15 minutes on a T4.

Needs `wf_cls.zip` (from `C:\Users\eru\Documents\Thesis`) in the Drive **FarmCam** folder. Afterwards download `whitefly_cls.onnx` and `whitefly_cls.pt` from Drive into `C:\Users\eru\Documents\Thesis\models_new\`.

In [ ]:
# Folders: wf_cls/{train,val,test}/{not_whitefly,whitefly} (make_wf_crops.py).
# Crops are 96x96: the Pi feeds the network exactly that size (wf_classifier.CROP).
import shutil
from ultralytics import YOLO
!rm -rf /content/data/wf_cls && mkdir -p /content/data
!unzip -q "$DRIVE/wf_cls.zip" -d /content/data
for split in ('train', 'val', 'test'):
    !echo {split}: $(ls /content/data/wf_cls/{split}/whitefly | wc -l) whitefly, $(ls /content/data/wf_cls/{split}/not_whitefly | wc -l) not_whitefly

YOLO('yolov8n-cls.pt').train(
    data='/content/data/wf_cls', imgsz=96, epochs=80, patience=0, batch=64, workers=2,
    fliplr=0.5, flipud=0.5, hsv_h=0.01, hsv_s=0.4, hsv_v=0.4, erasing=0.2,
    project=f'{DRIVE}/runs', name='wf_cls', exist_ok=True, plots=True)

best = f'{DRIVE}/runs/wf_cls/weights/best.pt'
metrics = YOLO(best).val(data='/content/data/wf_cls', split='test', imgsz=96, plots=False)
print('whitefly classifier test accuracy =', round(metrics.top1, 3))
path = YOLO(best).export(format='onnx', imgsz=96, opset=12, simplify=True, dynamic=False)
shutil.copy(path, f'{DRIVE}/whitefly_cls.onnx')
shutil.copy(best, f'{DRIVE}/whitefly_cls.pt')
!ls -la "$DRIVE"/whitefly_cls.*